In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import datetime

In [ ]:
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from matplotlib import pyplot as plt

In [ ]:
import requests

In [ ]:
from scipy.stats import sigmaclip
from scipy.signal import find_peaks
from scipy.ndimage import median_filter
# from pysr import PySRRegressor
from scipy.optimize import curve_fit

---

In [ ]:
from shared_matplotlib_utils import get_figure
from shared_matplotlib_utils.border_funcs import fix_borders

----

In [ ]:
#ENDPOINT = "http://homeassistant.local:8090"
ENDPOINT = "http://192.168.1.101:8090"

In [ ]:
frames = requests.get(ENDPOINT + "/frames/?filter=pull").json()
frames = frames.get("frames")
frames = {frame.get("id"): frame for frame in frames}

In [ ]:
assert frames

In [ ]:
frame_ids = list(frames.keys())
print(frame_ids[:10])

In [ ]:
frame_ids

---

## Get voltages, format, and remove outliers

In [ ]:
def remove_outliers(values, low_sigma=3.0, high_sigma=3.0):
    """
    Removes outliers from a 1D array or list using sigma clipping.
    
    Parameters:
    - values: list or numpy array of numeric values
    - low_sigma, high_sigma: sigma thresholds for clipping
    
    Returns:
    - outlier_indices: list of indices in the original array that were outliers
    """
    
    values = np.asarray(values)
    # sigmaclip returns the clipped values, and lower/upper thresholds
    clean_vals, low, high = sigmaclip(values, low=low_sigma, high=high_sigma)
    
    # Find indices of outliers
    outlier_mask = ~np.isin(values, clean_vals)
    outlier_indices = np.where(outlier_mask)[0].tolist()
    
    return outlier_indices

In [ ]:
battery_status = {}

for frame_id in frame_ids:
    readings = requests.get(ENDPOINT + f"/frames/{frame_id}/battery?limit=5000").json().get("readings")
    timestamps = [datetime.datetime.strptime(row.get("timestamp"), "%Y-%m-%dT%H:%M:%S.%f").timestamp() for row in readings]
    voltages = [row.get("voltage") for row in readings]

    timestamps = np.asarray(timestamps)
    voltages = np.asarray(voltages)

    order = np.argsort(timestamps)
    timestamps = timestamps[order]
    voltages = voltages[order]

    outliers = remove_outliers(voltages, low_sigma=5.0, high_sigma=5.0)

    if outliers:
        print("outliers:", [voltages[i] for i in outliers])
        timestamps = [value for i, value in enumerate(timestamps) if i not in outliers]
        voltages = [value for i, value in enumerate(voltages) if i not in outliers]
 
    battery_status[frame_id] = dict()
    battery_status[frame_id]["timestamp"] = timestamps
    battery_status[frame_id]["voltage"] = voltages

In [ ]:
# print(battery_status.get("e99c074f-da31-4582-8ef0-808adc02167d"))

In [ ]:
len(battery_status[frame_ids[0]].get("voltage"))

----

## Voltage over time

In [ ]:
def formatter_time_float(seconds, pos):
    """Automatically format time in seconds to nearest sec, mins, hours, days, years"""
    
    # Change to offset
    seconds = seconds - DAY_ZERO

    delta = datetime.timedelta(seconds=seconds)

    days = delta.days
    seconds = delta.seconds
    hours = seconds // 3600
    minutes = (seconds // 60) % 60

    return days
    
    if days > 0:
        return f"{days}d"

    if hours > 0:
        return f"{hours}h"

    if minutes > 0:
        return f"{minutes}m"

    return f"{seconds}s"

In [ ]:
fig, ax = get_figure()

DAY_ZERO = datetime.datetime.now().timestamp()

for frame_id in frame_ids:

    dates = battery_status.get(frame_id).get("timestamp")
    voltages = battery_status.get(frame_id).get("voltage")
    voltages = np.array(voltages)

    min_date = min(dates)
    if min_date < DAY_ZERO:
        DAY_ZERO = min_date

    ax.plot(dates, voltages, ".")

ax.xaxis.set_major_formatter(FuncFormatter(formatter_time_float))
fix_borders(ax)

----

## Per Day

In [ ]:
pdf = pd.concat(
    [pd.DataFrame(rows).assign(frame_id=group) for group, rows in battery_status.items()],
    ignore_index=True
)

In [ ]:
pdf

In [ ]:
pdf["day"] = pdf["timestamp"].map(lambda dt: datetime.datetime.fromtimestamp(dt).date())

In [ ]:
pdf

In [ ]:
groups_days = pdf.groupby(['frame_id', 'day'])['voltage'].apply(list)

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))

colors = plt.get_cmap('Set2').colors
colors = plt.get_cmap('tab10').colors

frame_ids = groups_days.index.get_level_values(0).unique()
days = sorted(groups_days.index.get_level_values(1).unique())

width = 0.8 / len(frame_ids)  # width of each box per group
x = np.arange(len(days))

zero_day = min(days)

for i, frame_id in enumerate(frame_ids):
    vals_per_day = []
    for day in days:

        # Safely get the voltage list for this frame_id/day
        try:
            vals = groups_days.loc[(frame_id, day)]
        except KeyError:
            vals = [3]
        vals_per_day.append(vals)
    
    positions = x + i*width  # shift boxes for this frame_id
    parts = ax.violinplot(vals_per_day, positions=positions, widths=width)

    # Color the violins
    color = colors[i]
    for pc in parts['bodies']:
        pc.set_facecolor(color)

# Set x-axis labels centered under grouped boxes

labels = [(day - zero_day).days for day in days]

ax.set_xticks(x + width*(len(frame_ids)-1)/2)
ax.set_xticklabels(labels)

# Legend
if False:
    handles = [Line2D([0], [0], color='w', markerfacecolor=colors[i], marker='s', markersize=10) for i in range(len(frame_ids))]
    ax.legend(handles, frame_ids, title="Frame ID", bbox_to_anchor=(1.05,1), loc='upper center')

fix_borders(ax)

# Split and prediction

In [ ]:
def split_discharges(timestamps, voltages, prominence=0.1, min_distance=10, min_segment_len=5):

    peak_indices, properties = find_peaks(voltages, prominence=prominence, distance=min_distance)

    segments = []
    boundaries = [0] + peak_indices.tolist() + [len(voltages)]
    
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        seg_t = timestamps[start:end]
        seg_v = voltages[start:end]
        if len(seg_t) >= min_segment_len:
            segments.append((seg_t, seg_v))
    
    return segments

In [ ]:
def remove_outliers_smooth(timestamps, voltages, window=5, threshold=0.05):
    smoothed = median_filter(voltages, size=window)
    mask = np.abs(voltages - smoothed) < threshold
    return timestamps[mask], voltages[mask]

In [ ]:
def get_sections(frame_id):
    timestamps = np.asarray(battery_status.get(frame_id).get("timestamp"))
    voltages = np.asarray(battery_status.get(frame_id).get("voltage"))
    results = split_discharges(timestamps, voltages)
    return results

In [ ]:
sections = []
for frame_id in frame_ids:
    _sections = get_sections(frame_id)
    for section in _sections:
        section = remove_outliers_smooth(*section)
        sections += [section]

print(len(sections[5][0]))

In [ ]:
fig, ax = plt.subplots(figsize=(20,6))
for stamps, voltages in sections:
    ax.plot(stamps, voltages, 'x')

In [ ]:
def filter_sections(sections, min_drop=0.15, min_points=10):
    """Keep only sections that are actual discharges."""
    good = []
    for seg_t, seg_v in sections:
        drop = seg_v[0] - seg_v[-1]
        if drop >= min_drop and len(seg_v) >= min_points:
            good.append((seg_t, seg_v))
    return good

In [ ]:
sections = filter_sections(sections)

In [ ]:
for i, (seg_t, seg_v) in enumerate(sections):
    print(f"Section {i}: v_start={seg_v[0]:.3f}, v_end={seg_v[-1]:.3f}, len={len(seg_v)}")

In [ ]:
def find_time_offset(base_t, base_v, seg_t, seg_v, v_lo=3.50, v_hi=3.70, n_grid=50):
    # Actual overlap
    v_lo = max(v_lo, base_v.min(), seg_v.min())
    v_hi = min(v_hi, base_v.max(), seg_v.max())
    
    if v_hi - v_lo < 0.05:
        return None
    v_grid = np.linspace(v_hi, v_lo, n_grid)
    # For base: sort by voltage ascending, average duplicate times per voltage
    # Since voltage decreases over time, flipping gives ascending voltage
    # But there may be non-monotonic noise, so sort explicitly
    base_order = np.argsort(base_v)
    bv_sorted = base_v[base_order]
    bt_sorted = base_t[base_order]
    
    # Remove duplicate voltages by taking mean time at each voltage
    bv_unique, indices = np.unique(bv_sorted, return_inverse=True)
    bt_unique = np.array([bt_sorted[indices == i].mean() for i in range(len(bv_unique))])
    
    base_t_interp = np.interp(v_grid, bv_unique, bt_unique)
    # Same for segment (relative time)
    seg_t_rel = seg_t - seg_t[0]
    seg_order = np.argsort(seg_v)
    sv_sorted = seg_v[seg_order]
    st_sorted = seg_t_rel[seg_order]
    
    sv_unique, indices = np.unique(sv_sorted, return_inverse=True)
    st_unique = np.array([st_sorted[indices == i].mean() for i in range(len(sv_unique))])
    
    seg_t_interp = np.interp(v_grid, sv_unique, st_unique)
    offsets = base_t_interp - seg_t_interp
    return np.median(offsets)

In [ ]:
def align_and_merge_sections(sections, v_lo=3.40, v_hi=3.70):
    # Sort by voltage RANGE (widest first), not by start voltage
    sections = sorted(sections, key=lambda s: s[1][0] - s[1][-1], reverse=True)
    
    # Baseline = section with widest range
    base_t = sections[0][0] - sections[0][0][0]  # t=0 at start
    base_v = sections[0][1]
    
    all_t = [base_t]
    all_v = [base_v]
    
    for seg_t, seg_v in sections[1:]:
        # Always align against the BASELINE only (not the growing merged data)
        offset = find_time_offset(base_t, base_v, seg_t, seg_v, v_lo, v_hi)
        if offset is None:
            print(f"No offset: seg {seg_v[0]:.3f}->{seg_v[-1]:.3f}")
            continue
        
        t_shifted = (seg_t - seg_t[0]) + offset
        all_t.append(t_shifted)
        all_v.append(seg_v)
    
    merged_t = np.concatenate(all_t)
    merged_v = np.concatenate(all_v)
    order = np.argsort(merged_t)
    return merged_t[order], merged_v[order]


In [ ]:
merged_section = align_and_merge_sections(sections)

In [ ]:
len(merged_section[0])

In [ ]:
fig, ax = plt.subplots(figsize=(20,6))
ax.plot(*merged_section, 'x')
fix_borders(ax)

In [ ]:
# Remove outliers and reset timestamps
merged_stamps, merged_volt = remove_outliers_smooth(*merged_section, window=10, threshold=0.1)

merged_volt, merged_stamps = remove_outliers_smooth(merged_volt, merged_stamps, window=10, threshold=150*60)

# Reset so t=0 is the start of the merged data
merged_stamps = merged_stamps - merged_stamps[0]

merged_section = [merged_stamps, merged_volt]

len(merged_stamps)

In [ ]:
fig, ax = plt.subplots(figsize=(20,6))
ax.plot(merged_stamps, merged_volt, 'x')
fix_borders(ax)

# Try to fit a function

In [ ]:
fit_stamps = np.abs(merged_stamps - np.max(merged_stamps))
fit_volt = np.array(merged_volt)

In [ ]:
coeffs = np.polyfit(fit_volt, fit_stamps, deg=4)
poly = np.poly1d(coeffs)
print(*coeffs)

In [ ]:
v_range = np.linspace(fit_volt.min()*0.95, fit_volt.max(), 200)
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(fit_volt, fit_stamps / 3600, 'x', alpha=0.3, color='steelblue', label='data')
ax.plot(v_range, poly(v_range) / 3600, '-', color='black', linewidth=2, label='poly deg=4')
ax.set_xlabel('Voltage (V)')
ax.set_ylabel('Time Hours')
ax.legend()
fix_borders(ax)

In [ ]:
def predict_remaining_seconds(timestamps, voltages, poly, V_dead=2.8):
    V_now = voltages[-1]
    t_now = timestamps[-1]
    if V_now <= V_dead:
        return 0.0
    if len(timestamps) < 2:
        t_remaining = poly(V_now) - poly(V_dead)
        return max(0.0, t_remaining)
    # Template values at each observed voltage
    p = poly(voltages)  # shape: (N,)
    # Fit: t_i = T - s * poly(V_i)
    # Rearrange as linear system: t_i = T * 1 + s * (-poly(V_i))
    # Design matrix: [1, -poly(V_i)]
    A = np.column_stack([np.ones(len(timestamps)), -p])
    # Least squares solve for [T, s]
    result, _, _, _ = np.linalg.lstsq(A, timestamps, rcond=None)
    T, s = result
    if s <= 0:
        t_remaining = poly(V_now) - poly(V_dead)
        return max(0.0, t_remaining)
    # Predicted time when voltage reaches V_dead
    t_dead = T - s * poly(V_dead)
    remaining = t_dead - t_now
    return max(0.0, remaining)

In [ ]:
example_i = 6
example_n = 10

In [ ]:
example_time = sections[example_i][0][-example_n:]
example_volt = sections[example_i][1][-example_n:]

print(example_volt[-1])

fig, ax = plt.subplots(figsize=(20, 6))
ax.plot(example_time, example_volt, 'x')
fix_borders(ax)


In [ ]:
death_time = predict_remaining_seconds(example_time, example_volt, poly)
print(death_time / (86400.0))